In [1]:
import jax
import os

jax_cache_dir = os.path.join(os.path.dirname(os.path.dirname(__vsc_ipynb_file__)), ".jax_cache")

jax.config.update("jax_compilation_cache_dir", jax_cache_dir)
jax.config.update("jax_persistent_cache_min_entry_size_bytes", -1)
jax.config.update("jax_persistent_cache_min_compile_time_secs", 0)

print('jax cache dir:', jax_cache_dir)

jax cache dir: /home/mat/ml/flax-qwen3/.jax_cache


In [2]:
from huggingface_hub import snapshot_download
from transformers import PreTrainedTokenizerFast

model_id = "Qwen/Qwen3-0.6B-Base"

snapshot_dir = snapshot_download(model_id)

tokenizer = PreTrainedTokenizerFast.from_pretrained(model_id)

print('snapshot_dl:', snapshot_dir)

PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

snapshot_dl: /home/mat/.cache/huggingface/hub/models--Qwen--Qwen3-0.6B-Base/snapshots/da87bfb608c14b7cf20ba1ce41287e8de496c0cd


In [3]:
import json
from pathlib import Path
from safetensors import safe_open

cfg = json.loads((Path(snapshot_dir) / "config.json").read_text())
L, N, K, H, D = cfg['num_hidden_layers'], cfg['num_attention_heads'], cfg['num_key_value_heads'], cfg['head_dim'], cfg['hidden_size']

weights = dict()
with safe_open(Path(snapshot_dir) / "model.safetensors", framework="flax") as f:
    for key in f.keys():
        weights[key] = f.get_tensor(key)


In [4]:
weights.keys()


dict_keys(['model.embed_tokens.weight', 'model.layers.0.input_layernorm.weight', 'model.layers.0.mlp.down_proj.weight', 'model.layers.0.mlp.gate_proj.weight', 'model.layers.0.mlp.up_proj.weight', 'model.layers.0.post_attention_layernorm.weight', 'model.layers.0.self_attn.k_norm.weight', 'model.layers.0.self_attn.k_proj.weight', 'model.layers.0.self_attn.o_proj.weight', 'model.layers.0.self_attn.q_norm.weight', 'model.layers.0.self_attn.q_proj.weight', 'model.layers.0.self_attn.v_proj.weight', 'model.layers.1.input_layernorm.weight', 'model.layers.1.mlp.down_proj.weight', 'model.layers.1.mlp.gate_proj.weight', 'model.layers.1.mlp.up_proj.weight', 'model.layers.1.post_attention_layernorm.weight', 'model.layers.1.self_attn.k_norm.weight', 'model.layers.1.self_attn.k_proj.weight', 'model.layers.1.self_attn.o_proj.weight', 'model.layers.1.self_attn.q_norm.weight', 'model.layers.1.self_attn.q_proj.weight', 'model.layers.1.self_attn.v_proj.weight', 'model.layers.10.input_layernorm.weight', 'm

In [5]:
weights['model.layers.0.mlp.up_proj.weight'].mean()

Array(-1.50204e-05, dtype=bfloat16)

In [6]:
import flax.linen as nn
import jax.numpy as j

class Qwen3Model(nn.Module):
  vocab_size: int
  hidden_size: int
  rms_norm_eps: float
  num_hidden_layers: int
  num_attention_heads: int
  num_key_value_heads: int
  head_dim: int
  rope_theta: float

  @nn.compact
  def __call__(self, x: jax.Array, pos=0):
    embeddings = nn.Embed(self.vocab_size, self.hidden_size)
    x = embeddings(x)

    T, C = x.shape

    for i in range(self.num_hidden_layers):
      lx = x

      # input norm
      lx_norm = nn.RMSNorm(self.rms_norm_eps, name=f"input_layernorm_{i}")(lx)
      
      # qkv projection
      q = nn.Dense(self.num_attention_heads * self.head_dim, use_bias=False, name=f"q_proj_{i}")(lx_norm)
      q = q.reshape(-1, self.num_attention_heads, self.head_dim)

      k = nn.Dense(self.num_key_value_heads * self.head_dim, use_bias=False, name=f"k_proj_{i}")(lx_norm)
      k = k.reshape(-1, self.num_key_value_heads, self.head_dim)

      v = nn.Dense(self.num_key_value_heads * self.head_dim, use_bias=False, name=f"v_proj_{i}")(lx_norm)
      v = v.reshape(-1, self.num_key_value_heads, self.head_dim)

      # QK norm
      q = nn.RMSNorm(self.rms_norm_eps, name=f"q_norm_{i}")(q)
      k = nn.RMSNorm(self.rms_norm_eps, name=f"k_norm_{i}")(k)
      
      # RoPE
      q = rope(q, self.rope_theta, pos)
      k = rope(k, self.rope_theta, pos)

      # attention
      attn_mask = j.tri(T, dtype=bool)
      attn_out = jax.nn.dot_product_attention(q, k, v, mask=attn_mask)
      attn_out = attn_out.reshape(T, -1)

      # out projection
      o = nn.Dense(self.hidden_size, use_bias=False, name=f"o_proj_{i}")(attn_out)
      lx += o

      # post-attention norm
      lx_norm = nn.RMSNorm(self.rms_norm_eps, name=f"post_attention_layernorm_{i}")(lx)
      
      # MLP
      gate = jax.nn.silu(nn.Dense(3 * self.hidden_size, use_bias=False, name=f"gate_proj_{i}")(lx_norm))
      up = nn.Dense(3 * self.hidden_size, use_bias=False, name=f"up_proj_{i}")(lx_norm)
      lx += nn.Dense(self.hidden_size, use_bias=False, name=f"down_proj_{i}")(gate * up)

      x = lx

    # logits
    x = nn.RMSNorm(self.rms_norm_eps, name='norm')(x)
    logits = embeddings.attend(x)

    return logits

def rope(x, theta, pos=0):
    T, N, H = x.shape
    positions = pos + j.broadcast_to(j.arange(T), [T])
    freq = 1.0 / (theta ** (j.arange(0, H, 2, dtype=j.float32) / H))
    inp = j.einsum('t,h->th', positions, freq, precision=jax.lax.Precision.HIGHEST)
    sin, cos = j.sin(inp).astype(x.dtype), j.cos(inp).astype(x.dtype)
    x1, x2 = x[:, :, :H//2], x[:, :, H//2:]
    sin, cos = sin[:, None, :], cos[:, None, :] # [B, T, 1, H/2]
    return j.concatenate([x1 * cos - x2 * sin, x2 * cos + x1 * sin], axis=-1)

In [7]:
# init model

model = Qwen3Model(
  vocab_size=cfg["vocab_size"],
  hidden_size=cfg["hidden_size"],
  rms_norm_eps=cfg["rms_norm_eps"],
  num_hidden_layers=cfg["num_hidden_layers"],
  num_attention_heads=cfg['num_attention_heads'],
  num_key_value_heads=cfg['num_key_value_heads'],
  head_dim=cfg['head_dim'],
  rope_theta=cfg['rope_theta'],
)

# params: convert the names from the Qwen3 snapshot to the Flax names
vars = {
  'params': {
    'Embed_0': {
      'embedding': weights['model.embed_tokens.weight']
    },
    'norm': {
      'scale': weights['model.norm.weight'],
    }
  }
}

for i in range(cfg["num_hidden_layers"]):
  vars['params'][f'input_layernorm_{i}'] = {
    'scale': weights[f'model.layers.{i}.input_layernorm.weight']
  }
  vars['params'][f'q_proj_{i}'] = {
    'kernel': weights[f'model.layers.{i}.self_attn.q_proj.weight'].transpose(1, 0),
  }
  vars['params'][f'k_proj_{i}'] = {
    'kernel': weights[f'model.layers.{i}.self_attn.k_proj.weight'].transpose(1, 0),
  }
  vars['params'][f'v_proj_{i}'] = {
    'kernel': weights[f'model.layers.{i}.self_attn.v_proj.weight'].transpose(1, 0),
  }
  vars['params'][f'q_norm_{i}'] = {
    'scale': weights[f'model.layers.{i}.self_attn.q_norm.weight'],
  }
  vars['params'][f'k_norm_{i}'] = {
    'scale': weights[f'model.layers.{i}.self_attn.k_norm.weight'],
  }
  vars['params'][f'o_proj_{i}'] = {
    'kernel': weights[f'model.layers.{i}.self_attn.o_proj.weight'].transpose(1, 0),
  }
  vars['params'][f'post_attention_layernorm_{i}'] = {
    'scale': weights[f'model.layers.{i}.post_attention_layernorm.weight'],
  }
  vars['params'][f'gate_proj_{i}'] = {
    'kernel': weights[f'model.layers.{i}.mlp.gate_proj.weight'].transpose(1, 0),
  }
  vars['params'][f'up_proj_{i}'] = {
    'kernel': weights[f'model.layers.{i}.mlp.up_proj.weight'].transpose(1, 0),
  }
  vars['params'][f'down_proj_{i}'] = {
    'kernel': weights[f'model.layers.{i}.mlp.down_proj.weight'].transpose(1, 0),
  }


In [8]:
print('Flax weights: ', jax.tree_util.tree_structure(vars))
print('Original weights: ', jax.tree_util.tree_structure(weights))

Flax weights:  PyTreeDef({'params': {'Embed_0': {'embedding': *}, 'down_proj_0': {'kernel': *}, 'down_proj_1': {'kernel': *}, 'down_proj_10': {'kernel': *}, 'down_proj_11': {'kernel': *}, 'down_proj_12': {'kernel': *}, 'down_proj_13': {'kernel': *}, 'down_proj_14': {'kernel': *}, 'down_proj_15': {'kernel': *}, 'down_proj_16': {'kernel': *}, 'down_proj_17': {'kernel': *}, 'down_proj_18': {'kernel': *}, 'down_proj_19': {'kernel': *}, 'down_proj_2': {'kernel': *}, 'down_proj_20': {'kernel': *}, 'down_proj_21': {'kernel': *}, 'down_proj_22': {'kernel': *}, 'down_proj_23': {'kernel': *}, 'down_proj_24': {'kernel': *}, 'down_proj_25': {'kernel': *}, 'down_proj_26': {'kernel': *}, 'down_proj_27': {'kernel': *}, 'down_proj_3': {'kernel': *}, 'down_proj_4': {'kernel': *}, 'down_proj_5': {'kernel': *}, 'down_proj_6': {'kernel': *}, 'down_proj_7': {'kernel': *}, 'down_proj_8': {'kernel': *}, 'down_proj_9': {'kernel': *}, 'gate_proj_0': {'kernel': *}, 'gate_proj_1': {'kernel': *}, 'gate_proj_10': 

In [9]:
weights[f'model.layers.{i}.self_attn.q_proj.weight'].shape

(2048, 1024)

In [10]:
# forward pass (random input)
key = jax.random.PRNGKey(0)
x = jax.random.randint(key, (1, 32), 0, 1<<15)
print('random input shape:', x.shape)
print('random input mean:', x.mean())
out = model.apply(vars, x[0])
print(out.shape)
print(out)


random input shape: (1, 32)
random input mean: 13089.719
(32, 151936)
[[5.6875 6.8125 -2.96875 ... -7.0625 -7.0625 -7.0625]
 [10.4375 9.5 8.25 ... 1.39062 1.39062 1.39062]
 [8.8125 10.25 13.375 ... -1.92188 -1.92188 -1.92188]
 ...
 [10.375 9.25 9.0625 ... 2.23438 2.23438 2.23438]
 [10.125 10.3125 10.0625 ... 2.125 2.125 2.125]
 [10.8125 10.75 10.5 ... 1.77344 1.77344 1.77344]]


In [11]:
# Generate text in a loop

text = 'The quick brown'
tokens = j.array(tokenizer(text)['input_ids'])

print(text)
for i in range(10):
  logits = model.apply(vars, tokens)

  next_token = logits[-1].argmax().item()
  next_text = tokenizer.decode(next_token)

  text += next_text
  tokens = j.concat([tokens, j.array([next_token])])
  print(next_text)


The quick brown
 fox
 jumps
 over
 the
 lazy
 dog
.
 The
 quick
 brown
